# HydraY NNUE — 160 superbatch: dove sta il tetto del budget?

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~4h, in quattro tappe da un'ora. **Non lasciare la scheda inattiva.**

### A cosa serve
Identico in tutto alla rete appena adottata — stesso dataset, stessa
architettura a 1024, stessi 4 king bucket, stessi 8 bucket di uscita, stesso WDL
0,3, stesso LR iniziale. L'unica variabile e' ancora il **budget**: 160
superbatch invece di 80.

### Perche'
Il budget e' stato la variabile nascosta **due volte**:

| | risultato |
|---|---|
| HalfKA a 20 SB | **perdeva** 11,3 Elo |
| HalfKA a 40 SB | +26,5 |
| 1024 a 40 SB (spedita con la 3.1.0) | +3,4 sulla 512 |
| **1024 a 80 SB** | **+29,4 ±11,1** sulla stessa a 40 |

Nessuno di questi run ha mai mostrato un tetto: ogni volta che il budget e'
stato alzato, ha reso. Quindi 160 non e' una formalita' — e' la domanda aperta
piu' economica che ci sia, perche' non costa NPS (l'architettura non cambia) e
non costa lavoro (nessun dato nuovo da generare o mescolare).

### Cosa aspettarsi, onestamente
Prima o poi il rendimento cala: 160 superbatch sono 12,5 epoche sullo stesso
1,25 miliardi di posizioni, e a un certo punto la rete smette di imparare e
inizia a memorizzare. Puo' benissimo essere questo il run che pareggia o perde.

Se pareggia o perde, **non e' un run sprecato**: e' la prima misura che dice
dove si ferma il budget, e sposta la domanda successiva sui **dati** (1024 su
~2,85B = v4 piu' i due batch di finali), che finora era ambigua perche' A5
l'aveva misurata a 40 SB — cioe' con un budget che oggi sappiamo insufficiente.

### Perche' non c'e' TEST_PATH
Nei notebook precedenti c'era, e non serviva a niente. Bullet al rev pinnato
accetta il TestDataset, stampa *"Validation data not currently implemented"* e
non legge mai la fetta. Nessun run ha mai prodotto una validation loss.
Rimosso, insieme al ritaglio dell'held-out: qui il giudice e' lo SPRT, e tanto
vale addestrare sul dataset intero.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va riletto e ristampato da Python: subprocess.run() senza capture
    # scrive sui file descriptor del KERNEL, che Colab non mostra nella cella.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset (le STESSE due parti di tutti i run su questa rete) ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]
P1, P2 = find('hydray_v6_part1.bin.zst'), find('hydray_v6_part2.bin.zst')
os.environ['P1'], os.environ['P2'] = P1, P2
print('parte 1:', P1, os.path.getsize(P1), 'byte')
print('parte 2:', P2, os.path.getsize(P2), 'byte')

NET_ID   = 'hydray-1024-160sb'
TOTAL_SB = 160            # il doppio della rete adottata: e' l'unica variabile
STAGE    = 40             # una tappa all'ora, quattro tappe
TRAINER  = '/content/th/nnue/trainer'
assert TOTAL_SB % STAGE == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
#
# NOTA sul branch: `nnue-1024` e' quello che ha di sicuro il trainer a 1024 su
# GitHub. Se nel frattempo hai pushato `dev` (che dalla 3.1.0 in poi e' anche
# lui a 1024), puoi cambiare BRANCH qui sotto: gli assert sono la vera difesa,
# non il nome del branch.
BRANCH = 'nnue-1024'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'i king bucket devono restare 4'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer.rs non e a 1024'
print(f'branch {BRANCH}, 1024 neuroni, 4 king bucket: ok')

In [ ]:
# --- decompressione in due parti (ogni zstd esce e libera la cache di Drive) ---
sh('apt-get -qq install -y zstd >/dev/null')

TOT, HALF = 41070409664, 20535204832
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~50 GiB')
assert free_gb > 52, 'disco insufficiente'

sh('zstd -d -T0 --long=27 -c "$P1" > /content/data.bin')
assert os.path.getsize('/content/data.bin') == HALF, 'parte 1 di taglia inattesa'
print('parte 1 ok'); sh('df -h /content | tail -1')

sh('zstd -d -T0 --long=27 -c "$P2" >> /content/data.bin')
SIZE = os.path.getsize('/content/data.bin')
assert SIZE == TOT, f'taglia finale inattesa: {SIZE}'
print('data.bin:', SIZE, 'byte =', SIZE // 32, 'posizioni')
print(f'{TOTAL_SB} superbatch = {TOTAL_SB*100_007_936/(SIZE//32):.1f} epoche')
sh('df -h /content | tail -1')

In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---
# Definisce come si lancia una tappa e come si salva il checkpoint. Sta in
# una cella separata apposta: la ripartenza salta il ciclo di training ma ha
# bisogno di queste due funzioni.
# Diviso in tappe SOLO per sopravvivenza: 4h e' abbastanza perche' Colab stacchi
# la sessione, e ogni tappa lascia un checkpoint su Drive. Il dataset e' lo
# stesso in tutte (nessuna fetta da scambiare) e lo schedule del learning rate
# keya su TOTAL_SB, quindi il calo cade dove cadrebbe in un run unico.
#
# ⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
# `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo la
# riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB senza
# salvare niente. Successo apparente, zero checkpoint -- e' costato un run
# intero. Qui si usa `env` esplicito proprio per rendere il legame visibile.
def stage_cmd(end, start=None, resume_from=None):
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero.
    Il mount di Drive scrive attraverso una cache: senza rilettura, un upload
    non completato passa per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)} byte'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)


In [ ]:
# --- training in quattro tappe da 40 superbatch ---
# NON eseguire questa cella in una ripartenza: rifarebbe le tappe gia' fatte.
# La cella qui sopra (helper delle tappe) invece va SEMPRE eseguita.
prev = None
for end in range(STAGE, TOTAL_SB + 1, STAGE):
    start = end - STAGE + 1
    print(f'\n===== tappa {start}-{end} di {TOTAL_SB} =====', flush=True)
    sh(stage_cmd(end, start, prev))
    # Se STAGE_END fosse stato ignorato, il trainer sarebbe andato oltre e
    # questo checkpoint non esisterebbe: l'assert dentro save_to_drive scatta
    # alla PRIMA tappa invece che a run finito.
    save_to_drive(end)
    prev = end

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# payload 6.326.288 + padding a 64 byte. La rete a 512 pesa 3.163.200: la
# taglia e' il controllo piu' rapido che l'architettura sia quella giusta.
assert 6326288 <= sz < 6326288 + 64, f'taglia {sz}: NON e la rete a 1024'
print('quantised.bin:', sz, 'byte — 1024 neuroni confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('RIFERIMENTO — la rete a 80 SB, quella da battere (misurata in locale):')
print('  startpos            52      mediogioco ~24 pezzi   975')
print('  KQvK               657      KRPvKR                  99')
print('  cavallo in piu     759      re attivi (finale)      21')
print('  donna in piu      1757      training loss      0.012767')
print()
print('E, un passo indietro, la rete a 40 SB (spedita con la 3.1.0):')
print('  startpos 71 | mediogioco 941 | KQvK 646 | loss 0.012920')
print()
print('Riporta la training loss finale e i sanity. Ma il verdetto e lo SPRT:')
print('i sanity a 80 SB sembravano quelli di una rete finita, e quella rete')
print('ha poi vinto di 29 Elo. Non fidarti del loro profilo.')
print('='*70)

In [ ]:
# --- RIPARTENZA (esegui SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle 1..6 (helper, Drive, Rust, clone, decompressione,
#      helper delle tappe)
#      -- il dataset va riscaricato e riscompattato, sono ~30 minuti;
#   2. NON eseguire la cella di training (rifarebbe le tappe gia' fatte);
#   3. metti DONE = ultimo superbatch completato che trovi su Drive
#      (cerca la cartella hydray-1024-160sb-<N> piu' alta) e lancia questa.
#
# stage_cmd/save_to_drive vivono in una cella loro (quella subito PRIMA del
# training), che va eseguita normalmente: nella ripartenza si salta solo la
# cella del ciclo di training.

DONE = 120   # ultimo superbatch completato e verificato su Drive

# Tappe piu' corte per il tratto finale: due sessioni su tre sono morte a meta'
# tappa, e con 20 una disconnessione costa 20 minuti invece di 40. Sicuro
# perche' il trainer salva un checkpoint ogni 10 superbatch (save_rate), quindi
# 140 e' un confine valido; e non cambia il risultato, perche' lo schedule del
# learning rate keya su TOTAL_SB, non sulla lunghezza delle tappe.
STAGE = 20
assert (TOTAL_SB - DONE) % STAGE == 0 and STAGE % 10 == 0

src = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
assert os.path.isdir(src), f'checkpoint non trovato su Drive: {src}'
# mkdir -p: su un clone fresco checkpoints/ non esiste, e senza questo `cp -r`
# creerebbe una cartella chiamata "checkpoints" AL POSTO del checkpoint.
os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
sh(f'cp -r {src} {TRAINER}/checkpoints/')
assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
print(f'ripartenza dal superbatch {DONE}, restano {TOTAL_SB-DONE} superbatch\n')

prev = DONE
for end in range(DONE + STAGE, TOTAL_SB + 1, STAGE):
    start = end - STAGE + 1
    print(f'\n===== tappa {start}-{end} di {TOTAL_SB} =====', flush=True)
    sh(stage_cmd(end, start, prev))
    save_to_drive(end)
    prev = end

## Come leggere il risultato

Il giudice e' lo SPRT testa a testa contro la rete a 80 superbatch, con lo
**stesso binario da entrambe le parti** e la candidata passata via `EvalFile`:
cosi' l'unica differenza fisica fra i due giocatori sono i pesi. Due misure
separate contro una baseline comune sommerebbero i loro errori.

```sh
cp "$HOME/Downloads/quantised.bin" nnue/data/hydray-1024-160sb.nnue
make prod && ./tuning/run_sprt.sh --snapshot
cd tuning && NEW_OPTS="EvalFile=$PWD/../nnue/data/hydray-1024-160sb.nnue" \
    MAXGAMES=8000 nohup ./run_sprt.sh > log_sprt_160sb.txt 2>&1 &
```

### Se vince
Il budget non ha ancora un tetto dopo tre raddoppi. Vale la pena chiedersi se
il vincolo vero non sia mai stato il budget in se', ma il **numero di epoche**
— cioe' se la strada non sia gia' quella dei dati, con piu' dati che
permettono piu' superbatch senza memorizzare.

### Se pareggia o perde
Il tetto e' fra 80 e 160, e la rete a 80 resta quella buona. A quel punto la
domanda successiva e' netta e finalmente non ambigua: **1024 su ~2,85 miliardi
di posizioni** (v4 da 2,75B piu' i due batch di finali — v4 da solo ha il
bucket 0 allo 0,23% e da solo farebbe perdere i ~35 Elo appena guadagnati),
con il budget alzato di conseguenza. A5 aveva bocciato i dati misurando a 40
superbatch, un budget che adesso sappiamo essere insufficiente: quel verdetto
non vale piu'.

### Una cosa da non rifare
I sanity eval **non predicono l'Elo**. A 80 superbatch KQvK era rimasto fermo a
657 contro un vero di circa 900, il che sembrava il ritratto di una rete a
corto di dati — e quella rete ha vinto di 29,4 Elo. Guardali per accorgerti di
un disastro (mirror rotto, valori assurdi, taglia sbagliata), non per prevedere
il risultato.
